In [28]:
import segmentation_models_pytorch as smp

aux_params={
    'classes': 2, # model output channels (number of classes in your dataset)
    'activation': 'sigmoid', # activation function
    'dropout': 0.5,
    'pooling': 'avg'

}

model = smp.Unet(
    encoder_name="resnet34",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=2,                      # model output channels
    aux_params=aux_params
)

model

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [29]:
import torch
fake_batch_img =torch.rand(size=(16, 3, 256, 256)) #16 images, 3 canali, 256x256 pixel

fake_batch_img

tensor([[[[0.7141, 0.7792, 0.4154,  ..., 0.7305, 0.5169, 0.6363],
          [0.9918, 0.7151, 0.7031,  ..., 0.6595, 0.9035, 0.8655],
          [0.2049, 0.0705, 0.3956,  ..., 0.8855, 0.6038, 0.8302],
          ...,
          [0.9359, 0.9815, 0.7782,  ..., 0.7956, 0.8936, 0.9461],
          [0.7927, 0.8993, 0.9100,  ..., 0.0605, 0.4364, 0.7483],
          [0.2614, 0.5093, 0.0309,  ..., 0.2535, 0.3464, 0.7882]],

         [[0.2564, 0.2846, 0.5210,  ..., 0.0586, 0.3627, 0.2982],
          [0.6610, 0.9808, 0.2758,  ..., 0.5777, 0.2373, 0.0432],
          [0.0037, 0.3139, 0.4290,  ..., 0.2021, 0.3918, 0.1248],
          ...,
          [0.6442, 0.8950, 0.6175,  ..., 0.1593, 0.2477, 0.9739],
          [0.5756, 0.5628, 0.4272,  ..., 0.2008, 0.8185, 0.7057],
          [0.0544, 0.8800, 0.1187,  ..., 0.8379, 0.4351, 0.9710]],

         [[0.6484, 0.9177, 0.4860,  ..., 0.4463, 0.9542, 0.1456],
          [0.3790, 0.1970, 0.4048,  ..., 0.3697, 0.9518, 0.7513],
          [0.9007, 0.1781, 0.0722,  ..., 0

In [30]:
seg_logits, cls_logits=model(fake_batch_img)
seg_logits.shape

cls_logits.shape


torch.Size([16, 2])

In [31]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import lightning as L
import segmentation_models_pytorch as smp

In [32]:
class FakeSegClsDataset(Dataset):
    def __init__(self, num_samples=100, image_size=256):
        self.num_samples = num_samples
        self.image_size = image_size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        image = torch.rand(3, self.image_size, self.image_size) #3 channels

        seg_mask = torch.randint(
            low=0,
            high=2, #binary segmentation mask with values 0 and 1
            size=(self.image_size, self.image_size)
        ).long() #long() is used to convert the tensor to integer type, which is required for segmentation masks

        cls_label = torch.randint(
            low=0,
            high=2,
            size=(1,) #the initial tensor has shape (1,), but we want it to be a scalar, so we use squeeze(0) to remove the extra dimension: tensor([0])-> tensor(0)    
        ).squeeze(0).long()

        return image, seg_mask, cls_label

In [33]:
#example: each sample in the dataset consists of a random image, a random binary segmentation mask, and a random binary classification label. The dataset can be used to test the model's forward pass and loss computation without needing real data.
dataset = FakeSegClsDataset(num_samples=100, image_size=256)
sample = dataset[0]

image, seg_mask, cls_label = sample

In [34]:
#I create the train and the validation loaders
train_dataset = FakeSegClsDataset(num_samples=100, image_size=256)
val_dataset = FakeSegClsDataset(num_samples=20, image_size=256)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True) #batch size of 8 means that the model will process 8 samples at a time.
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [35]:
#To check that the dataloader is working correctly
batch = next(iter(train_loader))
images, seg_masks, cls_labels = batch

print(images.shape)
print(seg_masks.shape)
print(cls_labels.shape)

torch.Size([8, 3, 256, 256])
torch.Size([8, 256, 256])
torch.Size([8])


In [36]:
class LitSegClsModel(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters() #saves the learning rate as a hyperparameter, which can be accessed later via self.hparams.lr

        aux_params = {
            "pooling": "avg",
            "dropout": 0.5,
            "activation": None,
            "classes": 2,
        }

        self.model = smp.Unet(
            encoder_name="resnet34",
            encoder_weights="imagenet",
            in_channels=3,
            classes=2,
            aux_params=aux_params,
        )

        self.seg_criterion = nn.CrossEntropyLoss()
        self.cls_criterion = nn.CrossEntropyLoss()

    def forward(self, x): #the forward method defines how the model processes input data and produces output
        seg_logits, cls_logits = self.model(x)
        return seg_logits, cls_logits

    def training_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch #batch is a tuple containing the input images, the corresponding segmentation masks, and the classification labels. 

        seg_logits, cls_logits = self(images) #self(images) is equivalent to self.forward(images), it calls the forward method of the model to get the segmentation and classification logits.

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(cls_logits, cls_labels)

        loss = seg_loss + cls_loss

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_seg_loss", seg_loss)
        self.log("train_cls_loss", cls_loss)

        return loss

    def validation_step(self, batch, batch_idx):
        images, seg_masks, cls_labels = batch

        seg_logits, cls_logits = self(images) #seg_logits and cls_logits are the raw output of the model, they represent the unnormalized scores for each class in the segmentation and classification tasks, respectively.

        seg_loss = self.seg_criterion(seg_logits, seg_masks)
        cls_loss = self.cls_criterion(cls_logits, cls_labels)

        loss = seg_loss + cls_loss

        cls_preds = torch.argmax(cls_logits, dim=1)
        cls_acc = (cls_preds == cls_labels).float().mean()

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_cls_acc", cls_acc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr) #self.parameters() returns an iterator over the model's parameters, which are the weights and biases that the optimizer will update during training. lr=self.hparams.lr sets the learning rate for the optimizer, which controls how much the model's parameters are updated in response to the computed gradients.
        return optimizer

In [37]:
model = LitSegClsModel(lr=1e-3) #I create a different model than before!! this is the model that will be trained using PyTorch Lightning.

In [38]:
trainer = L.Trainer(
    max_epochs=2,
    limit_train_batches=5, #only 5 batches for each epoch
    limit_val_batches=2,
    log_every_n_steps=1) #it means that the trainer will log the training metrics after every batch, which is useful for monitoring the training process closely

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [39]:
#Now the training starts
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
#I flops are 0 since the lighting module donesn't compute them. An additional module would be needed to compute them.

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ Unet             │ 24.4 M │ train │     0 │
│ 1 │ seg_criterion │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ cls_criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 24.4 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.4 M                                                                                               
Total estimated model params size (MB): 97                                                                         
Modules in train mode: 197                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:2
1: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing 
the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/Users/saratramontana/anaconda3/envs/test-seg/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/dat
a_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=2` reached.


Per collegarlo su github segui il seguente procedimento:
scrivi nel terminare: 
1) git init
2) git status
3) crea un file che si chiama .gitignore e incollaci dentro questa roba: 
__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
venv/
.env
lightning_logs/
wandb/
.DS_Store
4) torna nel file principale e scrivi git add .
